In [1]:
!pip install mlflow optuna imbalanced-learn dagshub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.

In [2]:
#!aws configure
import dagshub
dagshub.init(repo_owner="rohitbedse",repo_name="yt-comment-sentiment-analysis",mlflow=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=97240c1c-c0b4-4010-91e7-6c0dd161773f&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=5ab38d99bc3d4bfbc67689f780c55a660ad4c47284f8b0b13b1284adea1e5625




Accessing as rohitbedse

Initialized MLflow to track repo "rohitbedse/yt-comment-sentiment-analysis"

Repository rohitbedse/yt-comment-sentiment-analysis initialized!

In [3]:
import mlflow
# Step 2: Set up the MLflow tracking server
mlflow.set_tracking_uri("https://dagshub.com/rohitbedse/yt-comment-sentiment-analysis.mlflow")

In [5]:
# Set or create an experiment
mlflow.set_experiment("Exp 5 - ML Algos with HP Tuning With Chnaging Features")

<Experiment: artifact_location='mlflow-artifacts:/54b3eaefe0cb4e4d8246724361152697', creation_time=1775572221042, experiment_id='9', last_update_time=1775572221042, lifecycle_stage='active', name='Exp 5 - ML Algos with HP Tuning With Chnaging Features', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

In [6]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report,f1_score
from sklearn.svm import SVC
from sklearn.svm import LinearSVC
from imblearn.over_sampling import SMOTE
import mlflow
import mlflow.sklearn
import optuna
import numpy as np

In [7]:
df = pd.read_csv('/content/reddit_preprocessing.csv').dropna()
df.shape

(36662, 2)

In [8]:
# -----------------------------
# STEP 1: Label Mapping
# -----------------------------
df = df.dropna(subset=['category'])
df['category'] = df['category'].map({-1: 2, 0: 0, 1: 1})

# -----------------------------
# STEP 2: Show ORIGINAL label distribution
# -----------------------------
print("🔥 ORIGINAL LABELS:")
print(np.unique(df['category'], return_counts=True))

# -----------------------------
# STEP 3: TRAIN-TEST SPLIT (FIRST)
# -----------------------------
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['clean_comment'],
    df['category'],
    test_size=0.2,
    random_state=42,
    stratify=df['category']
)

# -----------------------------
# STEP 4: TF-IDF (FIT ONLY ON TRAIN)
# -----------------------------
vectorizer = TfidfVectorizer(ngram_range=(1,3), max_features=5000)

X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

# -----------------------------
# STEP 5: Show TRAIN distribution BEFORE SMOTE
# -----------------------------
print("\n📊 TRAIN LABELS BEFORE SMOTE:")
print(np.unique(y_train, return_counts=True))

# -----------------------------
# STEP 6: APPLY SMOTE ONLY ON TRAIN
# -----------------------------
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

# -----------------------------
# STEP 7: Show TRAIN distribution AFTER SMOTE
# -----------------------------
print("\n🚀 TRAIN LABELS AFTER SMOTE:")
print(np.unique(y_train_res, return_counts=True))

# -----------------------------
# STEP 9: MLflow logging
# -----------------------------
def log_mlflow(model_name, model, X_train, X_test, y_train, y_test):
    with mlflow.start_run():
        mlflow.set_tag("mlflow.runName", f"{model_name}_SMOTE_TFIDF_Bigram")
        mlflow.set_tag("experiment", "No_Leakage_Pipeline")

        mlflow.log_param("algo_name", model_name)
        mlflow.log_param("resampling", "SMOTE")
        mlflow.log_param("vectorizer", "TF-IDF Bigram")

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average='macro')

        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("f1_macro", f1)

        report = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in report.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        mlflow.sklearn.log_model(model, f"{model_name}_model")

# ================================
# Step 6: Optuna Objective (F1 Macro)
# ================================
def objective_svm(trial):
    C = trial.suggest_float('C', 1e-3, 10.0, log=True)

    model = LinearSVC(
        C=C,
        max_iter=5000,
        random_state=42
    )

    # ✅ SAME as XGBoost → split AFTER SMOTE
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train_res,
        y_train_res,
        test_size=0.2,
        random_state=42,
        stratify=y_train_res
    )

    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_val)

    return f1_score(y_val, y_pred, average='macro')


# ================================
# Step 7: Run Optuna + Log Best Model
# ================================
def run_optuna_experiment():
    study = optuna.create_study(direction="maximize")
    study.optimize(objective_svm, n_trials=30)

    best_params = study.best_params
    print("\n🏆 BEST PARAMS:", best_params)

    best_model = LinearSVC(
        C=best_params['C'],
        max_iter=5000,
        random_state=42
    )

    # ✅ FINAL TRAINING ON FULL SMOTE DATA
    log_mlflow(
        "LinearSVC",
        best_model,
        X_train_res,   # full SMOTE train
        X_test,        # untouched test
        y_train_res,
        y_test
    )


# ================================
# RUN
# ================================
run_optuna_experiment()

🔥 ORIGINAL LABELS:
(array([0, 1, 2]), array([12644, 15770,  8248]))

📊 TRAIN LABELS BEFORE SMOTE:
(array([0, 1, 2]), array([10115, 12616,  6598]))


[I 2026-04-08 04:04:05,725] A new study created in memory with name: no-name-5c687a59-cec5-4ce2-975d-4ec41d97012e



🚀 TRAIN LABELS AFTER SMOTE:
(array([0, 1, 2]), array([12616, 12616, 12616]))


[I 2026-04-08 04:04:07,183] Trial 0 finished with value: 0.8351504160120596 and parameters: {'C': 2.3683485098191057}. Best is trial 0 with value: 0.8351504160120596.
[I 2026-04-08 04:04:07,461] Trial 1 finished with value: 0.7132904902327204 and parameters: {'C': 0.011267787316821912}. Best is trial 0 with value: 0.8351504160120596.
[I 2026-04-08 04:04:07,645] Trial 2 finished with value: 0.6405297625518168 and parameters: {'C': 0.0010992109448831732}. Best is trial 0 with value: 0.8351504160120596.
[I 2026-04-08 04:04:07,920] Trial 3 finished with value: 0.6548607812748054 and parameters: {'C': 0.0022853051268416353}. Best is trial 0 with value: 0.8351504160120596.
[I 2026-04-08 04:04:08,530] Trial 4 finished with value: 0.8159525495693639 and parameters: {'C': 0.23529731741957574}. Best is trial 0 with value: 0.8351504160120596.
[I 2026-04-08 04:04:09,368] Trial 5 finished with value: 0.834696753491821 and parameters: {'C': 1.4220678414827506}. Best is trial 0 with value: 0.83515041


🏆 BEST PARAMS: {'C': 3.2676085231625587}


2026/04/08 04:04:44 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/08 04:05:11 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run LinearSVC_SMOTE_TFIDF_Bigram at: https://dagshub.com/rohitbedse/yt-comment-sentiment-analysis.mlflow/#/experiments/9/runs/869f26eae98248ba9f55a425d10a1b6b
🧪 View experiment at: https://dagshub.com/rohitbedse/yt-comment-sentiment-analysis.mlflow/#/experiments/9


In [9]:
best_params = {
    'C': 3.267
}

model = LinearSVC(
    **best_params,
    random_state=42,
)

model.fit(X_train_res, y_train_res)

y_pred = model.predict(X_test)

print("Test Macro F1:", f1_score(y_test, y_pred, average='macro'))
print("Test Weighted F1:", f1_score(y_test, y_pred, average='weighted'))

Test Macro F1: 0.8198613463267977
Test Weighted F1: 0.8342585136868015


In [10]:
import numpy as np

print("Unique predictions:", np.unique(y_pred))
print("Unique true labels:", np.unique(y_test))

Unique predictions: [0 1 2]
Unique true labels: [0 1 2]
